# RUL Prediction — Phase 1, Notebook 2
## C-MAPSS FD001: Preprocessing (RUL labels · Normalisation · Windowing)

**Dissertation:** Deep Learning-Based Remaining Useful Life Prediction for Industrial Rotating Equipment
**Author:** K Vatsav Sai Ram · BITS ID 2024TM05021

---

### Where we are
Notebook 1 loaded FD001 and identified the **14 informative sensors** to keep.
This notebook turns that raw data into the exact arrays an LSTM needs. Three jobs:

1. **RUL labelling** — give every training row a target: how many cycles until this engine fails
   (using the standard *piecewise-linear* target capped at 125).
2. **Normalisation** — scale the kept sensors to [0, 1], fitting the scaler on **training data only**
   (so no information leaks from the test set).
3. **Windowing** — slice each engine's history into fixed-length **sliding windows** of shape
   `(window, features)`, which is the sequence format an LSTM consumes.

By the end we save four ready-to-model arrays: `X_train, y_train, X_test, y_test`.


## 1. Setup — mount Drive, reload config from Notebook 1

We reuse the exact sensor choices saved by Notebook 1 (`config_FD001.json`) so the two notebooks
can never drift out of sync.

In [ ]:
import numpy as np, pandas as pd, os, json
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
plt.rcParams["figure.figsize"] = (11, 4)

# --- project paths: Colab+Drive if available, else a local folder ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = "/content/drive/MyDrive/RUL_Dissertation"
except ImportError:
    PROJECT_DIR = os.path.abspath(".")

DATA_DIR     = os.path.join(PROJECT_DIR, "data", "cmapss")
ARTIFACT_DIR = os.path.join(PROJECT_DIR, "artifacts")
PROC_DIR     = os.path.join(PROJECT_DIR, "processed")
MODEL_DIR    = os.path.join(PROJECT_DIR, "models")
for _d in (ARTIFACT_DIR, PROC_DIR, MODEL_DIR):
    os.makedirs(_d, exist_ok=True)
assert os.path.isdir(DATA_DIR), f"Place the C-MAPSS .txt files in {DATA_DIR}"
print("DATA_DIR:", DATA_DIR)

# reload the config saved by Notebook 1
with open(os.path.join(ARTIFACT_DIR, "config_FD001.json")) as f:
    cfg = json.load(f)

index_cols   = cfg["index_cols"]
setting_cols = cfg["setting_cols"]
sensor_cols  = cfg["sensor_cols"]
keep_sensors = cfg["keep_sensors"]
RUL_CAP      = cfg["rul_cap"]     # 125
all_cols     = index_cols + setting_cols + sensor_cols

print("Loaded config. Keeping", len(keep_sensors), "sensors. RUL cap =", RUL_CAP)

In [ ]:
def load_cmapss(split):
    df = pd.read_csv(f"{DATA_DIR}/{split}_FD001.txt", sep=r"\s+", header=None, names=all_cols)
    return df

train = load_cmapss("train")
test  = load_cmapss("test")
rul_truth = pd.read_csv(f"{DATA_DIR}/RUL_FD001.txt", sep=r"\s+", header=None, names=["RUL"])
print("train:", train.shape, "| test:", test.shape, "| rul_truth:", rul_truth.shape)

## 2. RUL labelling (training data)

### 2.1 The basic idea
In the training set each engine runs **to failure**, so for any row:

$$\text{RUL} = (\text{last cycle of that engine}) - (\text{current cycle})$$

At the failure row RUL = 0; one hundred cycles earlier RUL = 100.

In [ ]:
def add_linear_rul(df):
    df = df.copy()
    last_cycle = df.groupby("unit")["cycle"].transform("max")
    df["RUL"] = last_cycle - df["cycle"]
    return df

train = add_linear_rul(train)
print(train[["unit", "cycle", "RUL"]].head())
print("\nRUL range:", train["RUL"].min(), "to", train["RUL"].max())

### 2.2 Why we *cap* RUL — the piecewise-linear target

A brand-new engine isn't meaningfully "200 cycles from death" in any measurable way — early in life
the sensors look healthy and flat, and the true remaining life is essentially unknowable from them.
Asking the model to output a huge RUL for a healthy engine just teaches it to fit noise.

The **standard fix** in the C-MAPSS literature is a *piecewise-linear* RUL: hold RUL constant at a cap
(commonly **125**) while the engine is healthy, and only let it count down linearly once real
degradation is underway. This consistently improves results and is what published benchmarks use —
so we adopt it for fair comparison.

In [ ]:
def cap_rul(df, cap=RUL_CAP):
    df = df.copy()
    df["RUL"] = df["RUL"].clip(upper=cap)
    return df

train = cap_rul(train)

# Visualise the effect for one engine
u = 1
eng = train[train["unit"] == u]
plt.figure(figsize=(9, 4))
plt.plot(eng["cycle"], (eng["cycle"].max() - eng["cycle"]), "--", color="#B5651D", label="linear RUL (uncapped)")
plt.plot(eng["cycle"], eng["RUL"], color="#0E7C7B", lw=2, label=f"piecewise-linear RUL (cap={RUL_CAP})")
plt.title(f"RUL target for engine #{u}"); plt.xlabel("cycle"); plt.ylabel("RUL"); plt.legend()
plt.tight_layout(); plt.show()
print("Flat at the start (healthy), linear ramp down to 0 at failure.")

## 3. Normalisation (fit on TRAIN only)

Neural networks train far better when inputs are on a common scale. We use **min-max scaling** to map
each kept sensor to [0, 1].

**Critical for a valid experiment:** the scaler is fit on the **training data only**, then *applied* to
the test data. Fitting on the test set (or on both together) would leak information about the test
distribution into training — a classic methodological error that inflates results. Getting this right
is something an examiner may specifically check.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaler.fit(train[keep_sensors])                      # FIT on train only

train_scaled = train.copy()
test_scaled  = test.copy()
train_scaled[keep_sensors] = scaler.transform(train[keep_sensors])
test_scaled[keep_sensors]  = scaler.transform(test[keep_sensors])   # APPLY to test

print("After scaling, train sensor ranges (should be ~0..1):")
print(train_scaled[keep_sensors].describe().loc[["min", "max"]].round(2).T.head())

## 4. Sliding-window sequences — the LSTM input format

An LSTM reads a **sequence**, not a single row. So instead of feeding one cycle at a time, we feed a
**window** of the most recent `W` cycles and ask the model to predict the RUL at the *end* of that window.

For an engine with cycles `1..N` and window `W`, we create windows:
`[1..W] -> RUL at W`, `[2..W+1] -> RUL at W+1`, … `[N-W+1..N] -> RUL at N`.

Each window has shape `(W, n_features)`. Stacking many windows gives the 3-D array
`(n_windows, W, n_features)` that Keras LSTMs expect.

We use **W = 30** — a common choice for FD001. (It's a hyper-parameter we can tune later.)

In [ ]:
WINDOW = 30
FEATURES = keep_sensors
N_FEAT = len(FEATURES)

def make_training_windows(df, window=WINDOW, features=FEATURES):
    """Slide a window across each engine; label = RUL at the last row of the window."""
    Xs, ys = [], []
    for _, g in df.groupby("unit"):
        g = g.sort_values("cycle")
        vals = g[features].values
        rul  = g["RUL"].values
        n = len(g)
        if n < window:
            continue  # engine shorter than the window is skipped (rare in FD001)
        for start in range(0, n - window + 1):
            Xs.append(vals[start:start + window])
            ys.append(rul[start + window - 1])
    return np.array(Xs), np.array(ys)

X_train, y_train = make_training_windows(train_scaled)
print("X_train:", X_train.shape, "  (windows, timesteps, features)")
print("y_train:", y_train.shape)
print("Example: each sample is a", X_train.shape[1], "x", X_train.shape[2], "matrix -> one RUL value")

### 4.1 Test-set windows — one prediction per engine

For the test set we don't slide across the whole engine. Each test engine is truncated at some cycle,
and the task is to predict the RUL at that **final** cycle. So we take the **last `W` cycles** of each
test engine as its single window, and the label is the corresponding value from `RUL_FD001.txt`.

(If a test engine has fewer than `W` cycles, we left-pad by repeating its first row so every engine
still yields one `W`-length window.)

In [ ]:
def make_test_windows(df, rul_truth, window=WINDOW, features=FEATURES):
    Xs, ys = [], []
    units = df["unit"].unique()
    for i, u in enumerate(units):
        g = df[df["unit"] == u].sort_values("cycle")
        vals = g[features].values
        if len(vals) >= window:
            w = vals[-window:]
        else:
            pad = np.repeat(vals[:1], window - len(vals), axis=0)  # left-pad with first row
            w = np.vstack([pad, vals])
        Xs.append(w)
        ys.append(rul_truth.iloc[i, 0])
    return np.array(Xs), np.array(ys)

X_test, y_test = make_test_windows(test_scaled, rul_truth)
# The published protocol also caps the test RUL target at the same value
y_test = np.clip(y_test, a_min=None, a_max=RUL_CAP)
print("X_test:", X_test.shape, "  (one window per test engine)")
print("y_test:", y_test.shape)

## 5. Sanity checks & save

Quick checks that nothing is malformed (no NaNs, values in range, shapes consistent), then save the
four arrays to Drive so the modelling notebooks load them instantly without repeating this work.

In [ ]:
assert not np.isnan(X_train).any() and not np.isnan(X_test).any(), "NaNs in features!"
assert X_train.shape[1:] == X_test.shape[1:] == (WINDOW, N_FEAT), "window/feature mismatch!"
print("Feature value range  -> train:", X_train.min().round(3), "to", X_train.max().round(3))
print("RUL label range      -> train:", int(y_train.min()), "to", int(y_train.max()),
      "| test:", int(y_test.min()), "to", int(y_test.max()))

plt.figure(figsize=(11,3))
plt.subplot(1,2,1); plt.hist(y_train, bins=40, color="#0E7C7B"); plt.title("y_train (RUL) distribution"); plt.xlabel("RUL")
plt.subplot(1,2,2); plt.hist(y_test, bins=20, color="#10243B");  plt.title("y_test (RUL) distribution");  plt.xlabel("RUL")
plt.tight_layout(); plt.show()

In [ ]:
proc_dir = os.path.join(PROJECT_DIR, "processed")
os.makedirs(proc_dir, exist_ok=True)
np.savez_compressed(
    os.path.join(proc_dir, "fd001_windows.npz"),
    X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
    window=WINDOW, n_features=N_FEAT,
)
# also persist the fitted scaler + window setting into config for reproducibility
import joblib
joblib.dump(scaler, os.path.join(ARTIFACT_DIR, "scaler_FD001.joblib"))
cfg["window"] = WINDOW
with open(os.path.join(ARTIFACT_DIR, "config_FD001.json"), "w") as f:
    json.dump(cfg, f, indent=2)

print("Saved:")
print(" ", os.path.join(proc_dir, "fd001_windows.npz"))
print(" ", os.path.join(ARTIFACT_DIR, "scaler_FD001.joblib"))
print("Shapes -> X_train", X_train.shape, "| X_test", X_test.shape)

## 6. Summary & next step

**Done in this notebook:**
- Labelled training rows with **piecewise-linear RUL** (capped at 125).
- **Min-max normalised** the 14 sensors, fitting the scaler on **train only** (no leakage).
- Built **sliding windows** of shape `(30, 14)` — `X_train ≈ (17k+, 30, 14)`.
- Prepared **one window per test engine** with capped RUL labels.
- Saved `fd001_windows.npz` + the fitted scaler to Drive.

**Next — Notebook 3 (Baselines):**
Establish the "numbers to beat" with classical models (Random Forest / XGBoost) on the flattened
windows, evaluated with **RMSE** and the **PHM scoring function**. Then Notebook 4 builds the LSTM and
we compare — exactly the results your mid-semester report needs.
